# DP9 本番 C（Qwen2.5-7B-Instruct・L4）— 行 20〜29

> ## 🔑 先に許可しておくシークレット
> - **`GITHUB_TOKEN`（必須）**：リポジトリを決まったコミットで取ってきます。
> - `HF_TOKEN`（任意）：Qwen は利用許可のいらないモデルなので、なくても動きます。あればダウンロードが安定します。
>
> 左の鍵のマークで「ノートブックからのアクセス」をオンにしてから「すべてのセルを実行」してね。

**何をする？**：凍結したゲート `notes/dp9_qwen7b_replication_gate.md` どおりに、算数 40 問のうち **20〜29 の 10 問** を計算します。問題ごとに貪欲法の推論を 1 本作り、8 トークンおきの位置で 4 種類 × 32 本（と、3 つおきに free 32 本）の続きを引きます。問題の中身はどのセルも表示しません。

**使い方**
1. ランタイム → ランタイムのタイプを変更 → **L4**（ハイメモリは不要）。
2. **「すべてのセルを実行」**。最初の確認とモデルの準備に 15〜25 分、そのあと本番が **3〜5 時間くらい**（多めに見て 10 時間）。終わると通知が来て、ランタイムは自動で切れます。
3. **途中で切れたとき**：もう一度「すべてのセルを実行」。終わった仕事は Drive に残っていて、続きから再開します（合格した確認はやり直しません）。
4. この冊が終わったら、次は `DP9_QWEN_D.ipynb` を回してね。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。
- **エラーのとき**：通知の本文にエラーの文面（トレースバック）が入ります。全文は Drive の `DECISION_POINTS/_errors/` にも保存されるので、Claude が直接読めます。

- 通知の頭には 🚀（始まり）／⚠（エラー）／🎉（終わり）が付きます。

**止まる場所**：セル 1（取得・インストール・ゲートの照合）／セル 2（サインやサンプリングの照合）／セル 3（データの版・問題の重なり・モデルのファイル）／セル 4（E2・G1・B1・M1）／セル 6（**止める線**：土台の道の長さから見た見込みが 17.5 CU を超えたら、引き直しの前で止めます）。

In [ ]:
# Cell 1 — Drive, GPU check, the exact head, install, gate hash, folders.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, hashlib, datetime as dt

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。エラーのときはエラーの文面も送ります（鍵・トークンらしい文字は伏せ字。全文は Drive の DECISION_POINTS/_errors）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'DP9 本番 C（Qwen2.5-7B・行20〜29）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'GodPork - Pork DP9 C', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
import os, re as _re, traceback as _tb, datetime as _edt
ERR_SLUG = 'DP9_QWEN_C'   # Drive のエラーファイルの名前の頭
_SECRET_RE = _re.compile(r"sk-ant-[A-Za-z0-9_\-]+|github_pat_[A-Za-z0-9_]+|gh[pousr]_[A-Za-z0-9]{8,}|hf_[A-Za-z0-9]{8,}|x-access-token:[^@\s]+@")
def _scrub(s):
    s = _SECRET_RE.sub('[伏せ字]', s)
    return '\n'.join(l if len(l) <= 300 else l[:300] + ' …（長いので省略）' for l in s.splitlines())
def _error_text(result):
    e = result.error_in_exec if result.error_in_exec is not None else result.error_before_exec
    tb = ''.join(_tb.format_exception(type(e), e, e.__traceback__))
    first = (getattr(getattr(result, 'info', None), 'raw_cell', '') or '').strip().splitlines()[:1]
    return _scrub('セル: ' + (first[0][:100] if first else '?') + '\n' + tb)
def _save_error(text):
    base = '/content/drive/MyDrive'
    if not os.path.isdir(base):
        return None
    d = os.path.join(base, 'DECISION_POINTS', '_errors')
    os.makedirs(d, exist_ok=True)
    stem = ERR_SLUG + '_' + _edt.datetime.now(_edt.timezone(_edt.timedelta(hours=9))).strftime('%Y%m%d_%H%M%S')
    name, n = stem + '.txt', 1
    while os.path.exists(os.path.join(d, name)):
        n += 1
        name = stem + '_' + str(n) + '.txt'
    with open(os.path.join(d, name), 'w', encoding='utf-8') as fh:
        fh.write('[' + NB_LABEL + ']\n' + text)
    return 'DECISION_POINTS/_errors/' + name
def _short(text, limit=3300):
    b = text.encode('utf-8')
    if len(b) <= limit:
        return text
    return '…（前のほうは省略。全文は Drive）\n' + b[-limit:].decode('utf-8', 'ignore')
def _on_error(result):
    if result.error_in_exec is None and result.error_before_exec is None:
        return
    try:
        text = _error_text(result)
    except Exception as e2:
        text = 'エラーの文面を作れませんでした：' + type(e2).__name__
    try:
        where = _save_error(text)
    except Exception as e3:
        where = None
        text += '\n（Drive に保存できませんでした：' + type(e3).__name__ + '）'
    print('エラーの全文を通知に送ります' + ('（Drive：' + where + '）' if where else ''))
    notify('⚠ 止まりました：エラーが出ています。' + ('全文は Drive の ' + where if where else 'Drive には保存できませんでした')
           + '\n\n' + _short(text), tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

import tempfile
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
PART = "C"
drive.mount("/content/drive")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
assert gpu.returncode == 0 and gpu.stdout.strip(), "GPU のランタイムにしてね（L4）"
GPU_NAME = gpu.stdout.strip().split(",")[0]
print("GPU:", gpu.stdout.strip())
assert "L4" in GPU_NAME, "このノートブックは L4 で回す決まりです（ランタイムのタイプを L4 にしてね）"

EXPECTED_HEAD = "43ef025fdedc964048a596536ce180139e4f549f"
BRANCH = "decision-points"
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")
THEME = Path("/content/drive/MyDrive/DECISION_POINTS")
DP_ROOT = THEME / "dp9"
BASE = DP_ROOT / ("DP9_QWEN_" + PART + "_20261017")
UNITS = BASE / "units"
UNITS.mkdir(parents=True, exist_ok=True)
LOCKDIR = Path("/content/dp9_locks")
MODEL_DIR = Path("/content/qwen2.5-7b-instruct")
QP = Path("/content/drive/MyDrive/ULTIMAPORK/quality_pork")   # READ-ONLY: earlier studies' run identities
EARLIER = {
    "q3a": QP / "q3a_hightemp" / "QUALITY_PORK_Q3A_20261002", "q3b": QP / "q3b_topk" / "QUALITY_PORK_Q3B_20261002",
    "q4a": QP / "q4a_fence" / "QUALITY_PORK_Q4A_20261003", "q4b": QP / "q4b_sc" / "QUALITY_PORK_Q4B_T1_20261003",
    "q5": QP / "q5" / "QUALITY_PORK_Q5_A_20261004", "q6s": QP / "q6s" / "QUALITY_PORK_Q6S_A_20261004",
    "q8t": QP / "q8t" / "QUALITY_PORK_Q8T_A_20261005",
    "dp1": THEME / "dp1" / "DP1_1B_A_20261005", "dp2": THEME / "dp2" / "DP2_1B_A_20261006",
    "dp3": THEME / "dp3" / "DP3_1B_A_20261007", "dp4": THEME / "dp4" / "DP4_1B_A_20261008",
    "dp5": THEME / "dp5" / "DP5_GEMMA_A_20261008", "dp7": THEME / "dp7" / "DP7_GEMMA_A_20261010",
    "dp8": THEME / "dp8" / "DP8_GEMMA_A_20261015",
}
missing = [k for k, p in EARLIER.items() if not (p / "RUN_IDENTITY.json").exists() and k != "q8t"]
assert not missing, ("これまでの実行フォルダが見つかりません（問題の重なりの確認に使います）", missing)

def get_secret(name):
    try:
        v = userdata.get(name)
    except Exception:
        return None
    return str(v).strip() if v and str(v).strip() else None

token = get_secret("GITHUB_TOKEN") or get_secret("GH_TOKEN")
assert token, "Colab のシークレットに GITHUB_TOKEN を登録して、ノートブックからのアクセスをオンにしてね"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
os.chdir('/content')   # セル1 をやり直しても、消すフォルダの中にいないように（2026-10-14）
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "-q", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "-q", "--detach", EXPECTED_HEAD], check=True)
assert subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() == EXPECTED_HEAD
token = None
env.pop("GITHUB_TOKEN", None)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "transformers==4.57.6",
                    "huggingface_hub", "pyarrow", "scikit-learn", "scipy"], capture_output=True, text=True)
(BASE / "install_log.txt").write_text(r.stdout + "\n--- STDERR ---\n" + r.stderr)
assert r.returncode == 0, r.stderr[-4000:]
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import transformers, torch
assert transformers.__version__ == "4.57.6", transformers.__version__
from pork import dp9_qwen as d9
from pork import dp9_runner as rr
from pork import dp9_smoke as S
assert d9.gate_doc_sha256(REPO) == d9.GATE_DOC_SHA256, "DP9 のゲート文書が凍結したものと違います"
ROWS = d9.parts()[PART]

def atomic_json(p, value):
    p.parent.mkdir(parents=True, exist_ok=True)
    data = json.dumps(value, ensure_ascii=False, indent=1, sort_keys=True) + "\n"
    fd, tmp = tempfile.mkstemp(prefix=p.name + ".", suffix=".tmp", dir=p.parent)
    with os.fdopen(fd, "w", encoding="utf-8") as f:
        f.write(data); f.flush(); os.fsync(f.fileno())
    os.replace(tmp, p)
    assert json.loads(p.read_text(encoding="utf-8")) == value

def record_ok(name, check):
    p = BASE / name
    if not p.exists():
        return False
    v = json.loads(p.read_text())
    return v.get("gpu_name") == GPU_NAME and v.get("code_commit") == EXPECTED_HEAD and bool(check(v))
print("exact head:", EXPECTED_HEAD, "| part", PART, "rows", ROWS[0], "...", ROWS[-1], "| units on disk:", rr.progress(ROWS, UNITS))

In [ ]:
# Cell 2 — GPU vs plain-Python on synthetic logits with ties (vocab 2,000 and 152,064): the per-step signals, pure
# sampling, the alt first token, the w_tau choice.  No model.  Skipped if this part already has a passing record.
if record_ok("SAMPLER_PARITY.json", lambda v: v["pass"]):
    print("（前回の合格の記録があるので、飛ばします）")
else:
    import math, random, struct
    import torch as torch_mod
    from pork import pork_t_v1 as pt
    from pork import q4b_samplers as qb
    from pork import dp1_forks as dp
    Bg = pt.backend("torch", "cuda")
    def f32(x):
        return struct.unpack("f", struct.pack("f", x))[0]
    def synth(rng, vocab):
        xs = [f32(rng.gauss(0.0, rng.choice((1.0, 2.5, 4.0)))) for _ in range(vocab)]
        for i in range(0, vocab, 97):
            xs[i] = xs[0]
        if rng.random() < 0.3:
            j = rng.randrange(vocab); xs[j] = f32(xs[j] + 12.0)
        if rng.random() < 0.2:
            xs[1] = xs[0] = max(xs)
        if rng.random() < 0.4:
            m = max(xs)
            v = f32(m + math.log(d9.TAU) + rng.choice((0.0, 0.3, -0.3)))
            for j in rng.sample(range(2, vocab), 3):
                xs[j] = v
        return xs
    rng = random.Random(d9.PARITY_SEED)
    names = ("SIGNALS", "PURE", "ALT", "TAU")
    total = {k: 0 for k in names}; mism = {k: 0 for k in names}
    t0 = time.time()
    for vocab, steps in ((2000, 150), (152064, 8)):
        mg = mr = None
        for step in range(steps):
            xs = synth(rng, vocab)
            g = torch_mod.tensor(xs, dtype=torch_mod.float32, device="cuda")
            a, b = dp.step_signals(Bg, g, mg), dp.ref_step_signals(xs, mr)
            total["SIGNALS"] += 1
            if ((a["top1"], a["top2"]) != (b["top1"], b["top2"]) or abs(a["p1"] - b["p1"]) > 1e-9
                    or abs(a["p2"] - b["p2"]) > 1e-9 or abs(a["entropy"] - b["entropy"]) > 1e-9):
                mism["SIGNALS"] += 1
            mg, mr = a["mu_after"], b["mu_after"]
            u = rng.random()
            total["PURE"] += 1
            if tuple(int(v) for v in qb.pure_sample(Bg, g, u, 1.0)) != tuple(int(v) for v in qb.ref_pure(xs, u, 1.0)):
                mism["PURE"] += 1
            exa = b["top1"] if step % 2 == 0 else rng.randrange(vocab)
            u2 = rng.random() if step % 7 else 0.9999999999
            aa, ab = d9.alt_sample(Bg, g, u2, 1.0, exa), d9.ref_alt(xs, u2, 1.0, exa)
            total["ALT"] += 1
            if aa != ab or aa == exa:
                mism["ALT"] += 1
            ex = rng.sample(range(vocab), 4)
            ta, tb = d9.tau_choice(Bg, g, b["top1"], ex), d9.ref_tau_choice(xs, b["top1"], ex)
            total["TAU"] += 1
            if (ta["token"], ta["matched"]) != (tb["token"], tb["matched"]) or abs(ta["p"] - tb["p"]) > 1e-9 * tb["p"]:
                mism["TAU"] += 1
    parity = {"total": total, "mismatches": mism, "seconds": time.time() - t0, "gpu_name": GPU_NAME,
              "code_commit": EXPECTED_HEAD, "pass": all(v == 0 for v in mism.values())}
    atomic_json(BASE / "SAMPLER_PARITY.json", parity)
    print(json.dumps(parity, indent=1))
    assert parity["pass"], "サイン・サンプリングの照合が合いません — Claude に見せてね"
    print("PARITY PASS")

In [ ]:
# Cell 3 — data, the 40 DP9 items (earlier studies reconstructed and checked against their own run identities; no
# overlap with any earlier item, CR-1's candidates or DP8's items), the pinned model (file hashes checked), loaded once
# in this kernel (a model left in the runtime is removed first), and this part's run identity.  Stops before any
# benchmark token on any data problem.
import gc
for _old in ("model", "_single", "_batch", "PREFILL1", "EXPAND"):
    globals().pop(_old, None)
gc.collect()
import torch as torch_mod
torch_mod.cuda.empty_cache()
from pork import dp1_forks as dp
from pork import dp2_real_forks as d2
from pork import kv_cached_backend as kv
from pork import pork_t_v1 as pt
from pork.ultimapork_completion_c5a import PROMPTS as C5A_PROMPTS
try:
    rows_raw, DATA_ID = d2.load_rows_from_hub()
    ITEMS = d9.select_items(rows_raw)
    from pork import quality_pork_q3a_hightemp as q3a, quality_pork_q3b_topk as q3b, quality_pork_q4a_fence as q4a
    from pork import quality_pork_q4b_sc as q4b, quality_pork_q5 as q5, quality_pork_q6s as q6, quality_pork_q8t as q8
    from pork import dp3_flip as d3, dp4_weighting as d4w, dp5_gemma as d5, dp7_matched as d7, dp8_bridge as d8
    mods = {"q3a": q3a, "q3b": q3b, "q4a": q4a, "q4b": q4b, "q5": q5, "q6s": q6, "q8t": q8, "dp1": dp, "dp2": d2,
            "dp3": d3, "dp4": d4w, "dp5": d5, "dp7": d7, "dp8": d8}
    IDS = {k: json.loads((p / "RUN_IDENTITY.json").read_text()) for k, p in EARLIER.items() if (p / "RUN_IDENTITY.json").exists()}
    assert all(DATA_ID["dataset_revision"] == v["dataset_revision"] for v in IDS.values()), "data revision differs"
    earlier = {k: m.select_items(rows_raw) for k, m in mods.items()}
    for k in IDS:
        assert mods[k].items_sha256(earlier[k]) == IDS[k]["items_sha256"], (k, "item reconstruction does not match its run")
    OVERLAP = {k: dp.overlap_with(ITEMS, its) for k, its in earlier.items()}
    d9.check_cr1_candidates(rows_raw)
    OVERLAP["cr1_candidates"] = dp.overlap_with(ITEMS, d9.earlier_candidates(rows_raw))
    assert all(v == 0 for v in OVERLAP.values()), ("DP9 items overlap earlier items", OVERLAP)
    del rows_raw, earlier
    ITEMS_SHA = d9.items_sha256(ITEMS)
except Exception as exc:
    print("🛑 データの版・問題の重なりの確認で止まりました（ベンチマークのトークンはまだ作っていません）。理由:", repr(exc)[:600])
    raise SystemExit("Cell 3: data check failed before any benchmark token.")
print("dataset:", DATA_ID, "| items:", len(ITEMS), "| overlap:", OVERLAP, "| items_sha256:", ITEMS_SHA)

from huggingface_hub import snapshot_download
HF = get_secret("HF_TOKEN")
snapshot_download(d9.SPEC["id"], revision=d9.SPEC["revision"], local_dir=str(MODEL_DIR), token=HF,
                  allow_patterns=["*.json", "*.safetensors", "merges.txt", "vocab.json", "tokenizer*"])
for name, want in d9.MODEL_FILE_SHA256.items():
    h = hashlib.sha256()
    with open(MODEL_DIR / name, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 24), b""):
            h.update(chunk)
    assert h.hexdigest() == want, ("model file differs from the gate", name)
tokenizer = transformers.AutoTokenizer.from_pretrained(str(MODEL_DIR))
model = transformers.AutoModelForCausalLM.from_pretrained(str(MODEL_DIR), torch_dtype=torch_mod.bfloat16).to("cuda")
model.eval()
assert str(model.dtype) == "torch.bfloat16" and model.config._attn_implementation == d9.SPEC["attn_implementation"]
DEVICE = "cuda"
EOS = kv.eos_ids(tokenizer, model)
TAU_EXCLUDED = rr.excluded_ids(tokenizer, EOS)
VOCAB = int(model.get_output_embeddings().weight.shape[0])
_single, PREFILL1, EXPAND, _batch = rr.torch_pieces(model, torch_mod, DEVICE)
Bt = pt.backend("torch", DEVICE)
CHAT = lambda text: kv.chat_prompt_ids(tokenizer, text)
DECODE = lambda ids: tokenizer.decode(list(ids), skip_special_tokens=True)
prompt_sha = [hashlib.sha256(json.dumps(list(CHAT(d2.user_message(it["question"])))).encode()).hexdigest() for it in ITEMS]
identity = {"schema": "dp9-run-identity-v1", "part": PART, "rows": ROWS, "code_commit": EXPECTED_HEAD,
            "gate_doc_sha256": d9.GATE_DOC_SHA256, "items_sha256": ITEMS_SHA, "overlap": OVERLAP,
            "model_id": d9.SPEC["id"], "model_revision": d9.SPEC["revision"], "dtype": d9.SPEC["dtype"],
            "attn_implementation": model.config._attn_implementation, "model_local_path": str(MODEL_DIR),
            "dataset_revision": DATA_ID["dataset_revision"], "dataset_file": DATA_ID["dataset_file"],
            "eos_ids": list(EOS), "tau_excluded_ids": TAU_EXCLUDED, "vocab": VOCAB, "prompt_ids_sha256": prompt_sha,
            "gpu_name": GPU_NAME, "python": sys.version.split()[0], "torch": torch_mod.__version__,
            "transformers": transformers.__version__, "numpy": __import__("numpy").__version__}
id_path = BASE / "RUN_IDENTITY.json"
if id_path.exists():
    old = json.loads(id_path.read_text())
    drift = [k for k in ("part", "code_commit", "gate_doc_sha256", "items_sha256", "model_revision", "dataset_revision",
                         "eos_ids", "tau_excluded_ids", "vocab", "prompt_ids_sha256", "gpu_name", "attn_implementation")
             if old.get(k) != identity[k]]
    if drift:
        print("🛑 前回の記録と設定が違います:", drift, "— 画面を Claude に見せてね（何も書き換えていません）")
        raise SystemExit("identity drift — nothing was changed.")
    print("前回と同じ設定です（続きから再開します）")
else:
    atomic_json(id_path, identity)
print(json.dumps({k: v for k, v in identity.items() if k not in ("tau_excluded_ids", "prompt_ids_sha256")}, indent=1))

In [ ]:
# Cell 4 — pre-checks on the two consumed C5-A prompts (no benchmark tokens), with the gate's lines:
# E2 (cached step vs full replay, teacher forced, >= 29/32 steps and max |logit diff| <= 1.0), G1 (DP8's base-path
# generator vs plain greedy on the same cached logits, 64 tokens, identical; run twice, identical), B1 (32 identical
# rows vs one row, >= 29/32), M1 (the merged 128-row batch vs each prefix alone, >= 29/32 per group).
if record_ok("PRECHECKS.json", lambda v: v["pass"]):
    print("（前回の合格の記録があるので、飛ばします）")
else:
    from pork import quality_pork_q8t as q8
    from pork import dp8_bridge as d8
    c5a_ids = [kv.chat_prompt_ids(tokenizer, t) for _, t in C5A_PROMPTS]
    E2 = []
    for (name, _), ids in zip(C5A_PROMPTS, c5a_ids):
        step = _single()
        seq, past, new, agree, worst = list(ids), None, tuple(ids), 0, 0.0
        for _ in range(32):
            a, past = step(new, past)
            with torch_mod.no_grad():
                bfull = model(input_ids=torch_mod.tensor([seq], device=DEVICE), use_cache=False).logits[0, -1].float()
            worst = max(worst, float((a - bfull).abs().max()))
            tb = int(bfull.argmax()); agree += int(int(a.argmax()) == tb)
            seq.append(tb); new = (tb,)
        E2.append({"prompt": name, "agree_steps": agree, "max_abs_logit_diff": worst,
                   "pass": agree >= d9.E2_MIN_AGREE and worst <= d9.E2_MAX_DIFF})
    G1, BASES = [], []
    for (name, _), ids in zip(C5A_PROMPTS, c5a_ids):
        st, past, new, ref = _single(), None, tuple(ids), []
        for _ in range(d9.G1_STEPS):
            lg, past = st(new, past)
            tok = q8._greedy([float(v) for v in lg.tolist()]); ref.append(tok); new = (tok,)
            if tok in set(EOS):
                break
        g1 = rr.generate_base_tau(Bt, _single(), ids, EOS, TAU_EXCLUDED, len(ref))
        g2 = rr.generate_base_tau(Bt, _single(), ids, EOS, TAU_EXCLUDED, len(ref))
        BASES.append(g1)
        G1.append({"prompt": name, "steps": len(ref), "identical": g1["token_ids"] == ref,
                   "repeat_identical": g1["token_ids"] == g2["token_ids"]
                   and [s["tau_token"] for s in g1["steps"]] == [s["tau_token"] for s in g2["steps"]],
                   "pass": g1["token_ids"] == ref and g1["token_ids"] == g2["token_ids"]})
    ids = c5a_ids[0]
    bs, ss = _batch(), _single()
    bp, sp, bn, sn, agree = None, None, [tuple(ids)] * 32, tuple(ids), 0
    for _ in range(32):
        bl, bp = bs(bn, bp); sl, sp = ss(sn, sp)
        tok = int(sl.argmax()); agree += int(int(bl[0].argmax()) == tok)
        bn, sn = [(tok,)] * 32, (tok,)
    del bs, ss, bp, sp, bl, sl
    B1 = {"agree_steps": agree, "pass": agree >= d9.B1_MIN_AGREE}
    toks = BASES[0]["token_ids"]; t = min(32, len(toks) - 1)
    prefix = tuple(ids) + tuple(toks[:t])
    lg1, c1 = PREFILL1(prefix)
    w1, w2, wt = int(toks[t]), int(BASES[0]["steps"][t]["top2"]), int(BASES[0]["steps"][t]["tau_token"])
    alts = [int(d9.alt_sample(Bt, lg1, d9.stream("alt", 0, 0, j)[0], 1.0, w1)) for j in range(32)]
    firsts = [w1] * 32 + [w2] * 32 + alts + [wt] * 32
    reps, refs = [0, 32, 64, 96], {}
    for r0 in reps:
        st, past, new, seqr = _single(), None, prefix + (firsts[r0],), []
        for _ in range(32):
            lg, past = st(new, past); tk = int(lg.argmax()); seqr.append(tk); new = (tk,)
        refs[r0] = seqr
    bst, past = _batch(), EXPAND(c1, 128)
    rows_in = [(f,) for f in firsts]; ag = {r0: 0 for r0 in reps}
    for s_ in range(32):
        lgb, past = bst(rows_in, past)
        am = [int(v) for v in lgb.argmax(dim=-1).tolist()]
        for r0 in reps:
            ag[r0] += int(am[r0] == refs[r0][s_])
        rows_in = [((refs[reps[r // 32]][s_],) if firsts[r] == firsts[reps[r // 32]] else (am[r],)) for r in range(128)]
    del past, lgb, c1
    torch_mod.cuda.empty_cache()
    M1 = {"agree_steps": [ag[r0] for r0 in reps], "pass": all(v >= 29 for v in ag.values())}
    pre = {"E2": E2, "G1": G1, "B1": B1, "M1": M1, "gpu_name": GPU_NAME, "code_commit": EXPECTED_HEAD,
           "pass": all(r["pass"] for r in E2) and all(r["pass"] for r in G1) and B1["pass"] and M1["pass"]}
    atomic_json(BASE / "PRECHECKS.json", pre)
    print(json.dumps(pre, indent=1))
    assert pre["pass"], "本番前の確認が合いません — Claude に見せてね（ベンチマークのトークンはまだ作っていません）"
    print("PRE-CHECKS PASS")

In [ ]:
# Cell 5 — BENCHMARK, phase 1: the base paths of this part's 10 rows (DP8's generator).  Resumable.
for marker in BASE.glob("RUN_FAILED*.txt"):
    raise SystemExit(f"{marker.name} があります。Claude に見せてね")
cb = lambda r: rr.compute_base(r, ITEMS[r], B=Bt, chat_ids=CHAT, logits_fn_factory=_single, decode=DECODE, eos=EOS,
                               excluded=TAU_EXCLUDED)
cp = lambda r, i, b: rr.compute_point(r, i, b, ITEMS[r], B=Bt, chat_ids=CHAT, prefill1=PREFILL1, expand=EXPAND,
                                      batch_fn_factory=_batch, decode=DECODE, eos=EOS)
t0 = time.time()
n = rr.work(ROWS, ITEMS, units_dir=UNITS, lock_dir=LOCKDIR, code_commit=EXPECTED_HEAD, items_hash=ITEMS_SHA,
            compute_base_fn=cb, compute_point_fn=cp, only_bases=True, log=lambda m: print(m, flush=True))
print(f"base paths done ({n} computed now, {time.time() - t0:.0f}s) |", rr.progress(ROWS, UNITS))

In [ ]:
# Cell 6 — the gate's stop line, from the base paths' answer positions only (no answers, no q): the projected cost of
# this part's points (second smoke's per-step costs x third smoke's ratio 1.22).  Above 70 CU / 4 = 17.5 CU: stop
# before any point and talk (a design change would be a new amendment; the gate is not rewritten).
A_POS = rr.answer_positions(ROWS, units_dir=UNITS, code_commit=EXPECTED_HEAD, items_hash=ITEMS_SHA)
proj = d9.projected_cu(A_POS, S.SMOKE2_PER)
stop_rec = {"part": PART, "answer_positions": A_POS, "projection": proj, "gpu_name": GPU_NAME,
            "code_commit": EXPECTED_HEAD, "unix": time.time()}
atomic_json(BASE / f"STOPLINE_{int(time.time())}.json", stop_rec)
print(f"見込み：この冊の引き直し 約 {proj['hours']:.1f} 時間（{proj['cu']:.1f} CU、線は {proj['cap_cu']:.1f} CU）")
if proj["stop"]:
    notify("⚠ 止める線を超えました（見込み " + f"{proj['cu']:.1f}" + " CU）。引き直しの前で止めています。Claude に相談してね", tags="warning")
    raise SystemExit("stop line — no point was computed.")
print("STOP LINE OK")

In [ ]:
# Cell 7 — BENCHMARK, phase 2: every point of this part (merged 128-row batch + free 32 where i % 3 == 0).
# Resumable: after a disconnect, "Run all" again (finished units stay on Drive).
t0 = time.time()
n = rr.work(ROWS, ITEMS, units_dir=UNITS, lock_dir=LOCKDIR, code_commit=EXPECTED_HEAD, items_hash=ITEMS_SHA,
            compute_base_fn=cb, compute_point_fn=cp, log=lambda m: print(m, flush=True))
print(f"points done ({n} computed now, {(time.time() - t0) / 3600:.2f} h) |", rr.progress(ROWS, UNITS))

In [ ]:
# Cell 8 — post-hoc replay audit of this part's audit rows (gate: 5 items of 40, 3 points each; the merged batch
# rebuilt in the same order with rows j = 0..3 of every kind drawn by the numpy_stable_argsort reference).
AR = rr.audit_rows_in_part(PART)
cases = {c["row"]: c for c in rr.load_cases(AR, units_dir=UNITS, code_commit=EXPECTED_HEAD, items_hash=ITEMS_SHA)}
res = [rr.audit_row(r, ITEMS[r], cases[r], chat_ids=CHAT, step_fn_factory=_single, prefill1=PREFILL1, expand=EXPAND,
                    batch_fn_factory=_batch, decode=DECODE, eos=EOS, excluded=TAU_EXCLUDED, B=Bt, fast_reference=True)
       for r in AR]
audit = {"part": PART, "rows": AR, "per_row": res, "code_commit": EXPECTED_HEAD,
         "audit_reference": rr.AUDIT_REFERENCE, "diverged_rows": sum(r["diverged"] for r in res)}
atomic_json(BASE / f"AUDIT_{PART}.json", audit)
print("audit rows:", AR, "| diverged:", audit["diverged_rows"])
print("✅ この冊はおわり：" + PART + "（units " + str(rr.progress(ROWS, UNITS)) + "）")
notify('🎉 終わりました：DP9 本番 C（監査のずれ ' + str(audit['diverged_rows']) + ' 問）', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()